# Fraud detection – leakage-checked, cross-validated version
Changes vs. the original: API key removed from code, `id` never used as a feature, engineered features,
5-fold cross-validation instead of one lucky/unlucky split, and a threshold chosen by out-of-fold F1 instead of 0.5.

**Validation vs. submission format**
- PR-AUC / ROC-AUC are always computed from the true 0/1 labels and the **continuous scores**, never from thresholded 0/1 predictions.
- F1 / Recall are computed from 0/1 predictions made by thresholding those scores.
- `submission.csv` holds a **fraud score in [0,1]** (not 0/1). Its ranking gives PR-AUC, and it is shifted so that our chosen threshold sits at 0.5.

In [ ]:
# Install the TabPFN library
!pip install tabpfn lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 595.9/595.9 kB 48.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from google.colab import files

# This opens a file upload button on your screen
uploaded = files.upload()

# Get the file name automatically (or replace 'your_fraud_data.csv' with your exact filename)
filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)

# Quick check: see the first 5 rows and column names
print("Dataset Shape:", df.shape)
df.head()

Saving Track 2 Training Dataset.csv to Track 2 Training Dataset.csv
Dataset Shape: (20000, 12)


,id,transaction_amount,transaction_hour,merchant_category,country,transaction_channel,transactions_last_24h,spend_last_24h,account_age,new_device,transactions_last_1h,fraud
0,FR016993,55.03,16,retail,SG,ecommerce,27.0,1791.34,1192.0,0.0,5.0,0
1,FR002433,20.75,1,gaming,SG,mobile_app,3.0,217.96,3073.0,0.0,0.0,0
2,FR009253,21.51,9,entertainment,SG,mobile_app,3.0,62.71,985.0,0.0,0.0,0
3,FR004881,813.98,16,utilities,GB,mobile_app,2.0,110.48,653.0,0.0,1.0,0
4,FR014830,13.54,20,food_delivery,SG,card_present,4.0,114.98,79.0,0.0,0.0,0


In [ ]:
import os
# SECURITY: never paste the token into the notebook. In Colab, add it under the key icon (Secrets) as TABPFN_TOKEN.
from google.colab import userdata
os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_TOKEN")
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

In [ ]:
import numpy as np

target_column = "fraud"
CATS = ["merchant_category", "country", "transaction_channel"]

# Adds per-row features (uses only that row's own values, so no label or cross-row leakage)
def engineer(d):
    d = d.copy()
    d["log_amt"] = np.log1p(d.transaction_amount)
    d["log_spend"] = np.log1p(d.spend_last_24h)
    d["log_age"] = np.log1p(d.account_age)
    d["hour_sin"] = np.sin(2 * np.pi * d.transaction_hour / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d.transaction_hour / 24)
    d["night"] = d.transaction_hour.isin([23, 0, 1, 2, 3, 4]).astype(int)
    d["burst"] = (d.transactions_last_1h >= 4).astype(float)
    d["high24"] = (d.transactions_last_24h >= 11).astype(float)
    d["young"] = (d.account_age <= 30).astype(float)
    d["amt_to_avg_ticket"] = d.log_amt - np.log1p(d.spend_last_24h / d.transactions_last_24h.clip(lower=1))
    d["share_1h"] = d.transactions_last_1h / d.transactions_last_24h.clip(lower=1)
    d["risky_merchant"] = d.merchant_category.isin(["luxury", "cash_transfer", "electronics"]).astype(int)
    d["night_x_device"] = d.night * d.new_device
    d["n_flags"] = d[["burst", "high24", "young", "new_device", "night", "risky_merchant"]].fillna(0).sum(axis=1)
    for c in CATS:
        d[c] = d[c].fillna("Unknown").astype("category")
    return d

# --- DROP UNNECESSARY ID COLUMNS --- (id is a random label; checked: it carries no fraud signal)
X = engineer(df.drop(columns=["id", target_column]))
y = df[target_column].values
NUM_COLS = [c for c in X.columns if c not in CATS]
print(f"Features being used ({X.shape[1]}): {list(X.columns)}")

Features being used (24): ['transaction_amount', 'transaction_hour', 'merchant_category', 'country', 'transaction_channel', 'transactions_last_24h', 'spend_last_24h', 'account_age', 'new_device', 'transactions_last_1h', 'log_amt', 'log_spend', 'log_age', 'hour_sin', 'hour_cos', 'night', 'burst', 'high24', 'young', 'amt_to_avg_ticket', 'share_1h', 'risky_merchant', 'night_x_device', 'n_flags']


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import lightgbm as lgb
from tabpfn import TabPFNClassifier

# Logistic regression; imputation/scaling/encoding are fitted inside the pipeline, so only on training rows
def make_logit():
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median", add_indicator=True), StandardScaler()), NUM_COLS),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATS),
    ])
    return make_pipeline(pre, LogisticRegression(C=0.3, max_iter=3000))

# Small, heavily regularised LightGBM (handles NaN and pandas categories natively)
def make_lgb():
    return lgb.LGBMClassifier(n_estimators=400, learning_rate=0.02, num_leaves=7, min_child_samples=60,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=5, verbose=-1)

# TabPFN with the same engineered features
def make_tabpfn():
    return TabPFNClassifier(ignore_pretraining_limits=True)

MODELS = {"logistic": make_logit, "lightgbm": make_lgb, "tabpfn": make_tabpfn}

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score

# 5-fold CV: every row is tested exactly once, so the score uses all 353 frauds instead of 71
# Probability of the fraud class (looked up in classes_, not assumed to be column 1)
def p_fraud(model, X_):
    return model.predict_proba(X_)[:, list(model.classes_).index(1)]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
oof = {name: np.zeros(len(y)) for name in MODELS}   # out-of-fold predictions
fold_scores = {name: [] for name in MODELS}

for fold, (tr, te) in enumerate(skf.split(X, y)):
    for name, make in MODELS.items():
        model = make()
        model.fit(X.iloc[tr], y[tr])
        oof[name][te] = p_fraud(model, X.iloc[te])
        fold_scores[name].append(average_precision_score(y[te], oof[name][te]))
    print(f"fold {fold} done")

# PR-AUC / ROC-AUC below compare true 0/1 labels with continuous scores (no thresholding)
# Rank-average blend of the three models (ranks make different probability scales comparable)
oof["blend"] = sum(pd.Series(oof[m]).rank(pct=True).values for m in MODELS) / len(MODELS)

for name, p in oof.items():
    extra = f"  fold std={np.std(fold_scores[name]):.3f}" if name in fold_scores else ""
    print(f"{name:9s} PR-AUC={average_precision_score(y, p):.4f}  ROC-AUC={roc_auc_score(y, p):.4f}{extra}")

tabpfn-v3.5-20260909.safetensors: reconstructing file:   0%|          |  0.00B /  876MB            

tabpfn-v3.5-20260909.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

fold 0 done
fold 1 done
fold 2 done
fold 3 done
fold 4 done
logistic  PR-AUC=0.2236  ROC-AUC=0.7792  fold std=0.051
lightgbm  PR-AUC=0.2098  ROC-AUC=0.7725  fold std=0.049
tabpfn    PR-AUC=0.2323  ROC-AUC=0.7805  fold std=0.059
blend     PR-AUC=0.2245  ROC-AUC=0.7815


In [ ]:
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score

# Blend is within CV noise of the best single model (fold std ~0.05) and more stable, so we use it
best = "blend"

# Choose the share of rows to flag by OOF F1. The share is tuned on these same OOF rows, so this F1 is slightly optimistic.
shares = np.arange(0.005, 0.0605, 0.0025)
f1s = [f1_score(y, (oof[best] >= np.quantile(oof[best], 1 - s)).astype(int)) for s in shares]
best_share = shares[int(np.argmax(f1s))]
threshold = np.quantile(oof[best], 1 - best_share)
y_pred = (oof[best] >= threshold).astype(int)   # 0/1 labels, used only for F1 / Recall

print(f"Flagging top {best_share:.2%} of transactions ({y_pred.sum()} alerts)")
print(classification_report(y, y_pred, digits=3))

In [ ]:
# Submission score: the rank blend is not a calibrated probability, and if F1/Recall are computed at the
# default 0.5 cut-off, a raw rank score would flag ~50% of rows. This strictly increasing map keeps the
# ranking (so PR-AUC is unchanged) and puts our threshold at 0.5: flagged rows -> [0.51, 1], others -> [0, 0.49].
def to_submission_score(score, thr):
    score = np.asarray(score, dtype=float)
    lo, hi = score.min(), score.max()
    below = 0.49 * (score - lo) / max(thr - lo, 1e-12)
    above = 0.51 + 0.49 * (score - thr) / max(hi - thr, 1e-12)
    return np.clip(np.where(score >= thr, above, below), 0, 1)

# Score the OOF predictions exactly as the organisers will score submission.csv
oof_sub = to_submission_score(oof[best], threshold)
assert np.isclose(average_precision_score(y, oof_sub), average_precision_score(y, oof[best])), "ranking changed"
pred_05 = (oof_sub >= 0.5).astype(int)
assert (pred_05 == y_pred).all(), "0.5 cut-off does not reproduce the chosen threshold"

print(f"Submission-format OOF check: PR-AUC = {average_precision_score(y, oof_sub):.4f} (on scores), "
      f"F1@0.5 = {f1_score(y, pred_05):.3f}, Recall@0.5 = {recall_score(y, pred_05):.3f}, "
      f"Precision@0.5 = {precision_score(y, pred_05):.3f}")

## Predict the test set and write `submission.csv`
Train every model on **all** training rows, predict the test file, and save `id,prediction`.
`prediction` is a fraud score in [0,1] (not 0/1). Its ranking gives PR-AUC, and scores ≥ 0.5 are our fraud flags for F1/Recall.

In [ ]:
# This opens a file upload button: pick the TEST csv (the one without the fraud column)
uploaded_test = files.upload()
test_filename = list(uploaded_test.keys())[0]
test_df = pd.read_csv(test_filename)

print("Test Shape:", test_df.shape)
assert "fraud" not in test_df.columns, "This looks like the training file - upload the test file instead"

# Same feature steps as training (id kept aside, never used as a feature)
X_test_final = engineer(test_df.drop(columns=["id"]))

# Make test categories use exactly the same category list as training
for c in CATS:
    X_test_final[c] = pd.Categorical(X_test_final[c].astype(str), categories=X[c].cat.categories)
X_test_final = X_test_final[X.columns]

In [ ]:
# Train each model on all 20,000 training rows, then score the test rows
test_scores = {}
for name, make in MODELS.items():
    model = make()
    model.fit(X, y)
    test_scores[name] = p_fraud(model, X_test_final)
    print(f"{name} trained and predicted")

# Rank-average blend, same as in cross-validation
test_scores["blend"] = sum(pd.Series(test_scores[m]).rank(pct=True).values for m in MODELS) / len(MODELS)

In [ ]:
# Flag the same share of test rows as in CV (best_share). Note: test prevalence is unknown and may be higher (see shift notes).
test_threshold = np.quantile(test_scores[best], 1 - best_share)
test_prob = to_submission_score(test_scores[best], test_threshold)

submission = pd.DataFrame({
    "id": test_df["id"],
    "prediction": test_prob,   # continuous score in [0,1]; >= 0.5 means flagged as fraud
})

# Sanity checks before writing
assert len(submission) == len(test_df) and submission["id"].is_unique
assert submission["prediction"].notna().all() and submission["prediction"].between(0, 1).all()
assert submission["prediction"].nunique() > 2, "prediction looks binary - PR-AUC needs continuous scores"
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"Rows: {len(submission)}, flagged (>= 0.5): {(test_prob >= 0.5).sum()} ({(test_prob >= 0.5).mean():.2%})")

In [ ]:
# Download submission.csv to your computer
files.download("submission.csv")